[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Processor Datapath and Control** {#processor-datapath-and-control}

Chapter 05 described instructions as architectural promises: `add` changes one register, `sw` changes memory, `beq` may change the next program counter, and `jal` changes both a link register and the program counter. Those statements describe **what** software must observe. A processor must still decide **how** gates, state elements, memories, and wires will produce exactly those changes.

This chapter develops that implementation bridge using a deliberately small RV32I subset. The design supports representative register arithmetic, loads, stores, conditional branches, and `jal`. It is a teaching microarchitecture, not the only valid RISC-V implementation. A tiny single-cycle core, a multi-cycle controller, a pipelined in-order core, and a speculative out-of-order core can all implement the same ISA while arranging the hardware very differently.

### **From ISA Semantics to a Processor** {#from-isa-semantics-to-a-processor}

An instruction is best understood as a state transition. Let the programmer-visible state before an instruction be

$$
S=(PC, R, M),
$$

where $PC$ is the current instruction address, $R$ is the register file, and $M$ is memory. Executing instruction $I$ must produce

$$
S^{+}=F_I(S).
$$

- $S^{+}$ is the architectural state after the instruction commits.
- $F_I$ is the transition required by instruction $I$.
- The subscript $I$ matters because `add`, `lw`, `sw`, `beq`, and `jal` update different parts of the state.

For `add x5, x6, x7`, RV32I requires

$$
R^{+}[5]=(R[6]+R[7])\bmod 2^{32},\qquad PC^{+}=PC+4,
$$

while all other registers and memory remain unchanged. The modulo operation means the 32-bit result wraps naturally when an unsigned sum exceeds $2^{32}-1$; the same bit pattern can later be interpreted as signed or unsigned by another instruction.

![An ISA statement is decoded into control decisions, values flow through the datapath, and selected state changes commit at the clock edge.](assets/isa-semantics-to-hardware.svg){fig-align="center" width="100%"}

The implementation divides the work into two cooperating parts:

- The **datapath** contains state elements and value-transforming hardware: the PC, register file, memories, ALU, adders, comparators, and multiplexers.
- The **control unit** interprets instruction bits and current conditions, then produces enables and selector values telling the datapath which operation is valid.

A useful analogy is a rail network. The datapath provides tracks, stations, and switches; control selects a route. A train still carries the value, but a control signal determines whether it reaches the ALU, data memory, register file, or next-PC input.

| ISA requirement | Datapath obligation | Control decision |
|---|---|---|
| read `rs1` and `rs2` | expose two register-file read ports | select encoded register indices |
| compute an ALU result | route operands to the ALU | select register or immediate and choose operation |
| load or store | present an effective address to data memory | enable memory action and choose write-back source |
| branch or jump | compute candidate next addresses | select sequential or target PC |
| commit exactly one legal result | provide edge-triggered state inputs | assert only the permitted write enables |

This separation is the central design discipline. Values may be present on many wires simultaneously, but only enabled state elements make those values architecturally visible.

### **Core Datapath Components** {#core-datapath-components}

A processor combines **state** and **combinational transformation**. State remembers values across clock edges. Combinational blocks calculate candidate values during a cycle but remember nothing after their inputs change. Memories form interfaces whose timing depends on the implementation: a simple teaching model may treat reads as combinational, while real caches and external memory use request/response protocols and variable latency.

![Processor components have distinct roles: state elements remember, combinational blocks transform, memories supply instructions or data, and control steers the connections.](assets/datapath-component-roles.svg){fig-align="center" width="100%"}

The distinction explains why a datapath diagram contains many arrows but only a few architectural updates. During one cycle, `PC+4`, a branch target, an ALU sum, and a memory output may all be calculated. Multiplexers choose candidates, and write-enable signals decide which selected candidates are captured at the edge.

| Component type | Typical examples | Stores information across cycles? | Main question it answers |
|---|---|---:|---|
| architectural state | PC, integer registers, memory | yes | what can software observe? |
| internal state | instruction register, `ALUOut`, memory-data register | yes | what must a multi-cycle implementation remember? |
| combinational transformation | ALU, comparator, immediate generator, adder | no | what candidate value should be calculated? |
| steering | multiplexer, decoder, enable logic | no | which candidate or action is selected? |

The following subsections inspect the five components that appear in almost every scalar datapath trace.

#### **Program Counter** {#program-counter}

The **program counter** is a register containing the address used to fetch the current instruction. It is called a counter because straight-line code usually advances by the instruction length, but it is more accurately a **next-instruction-address register**: branches, jumps, traps, and returns can replace ordinary counting.

For the base 32-bit instructions used here, the sequential candidate is

$$
PC_{seq}=PC+4.
$$

The constant 4 is measured in bytes because memory is byte addressed and each base instruction occupies four bytes. A simplified next-PC rule is

$$
PC^{+}=\begin{cases}
PC+Imm, & \text{if a branch is taken or `jal` executes},\\
PC+4, & \text{otherwise}.
\end{cases}
$$

- $PC^{+}$ is the value captured by the PC at the next active clock edge.
- $Imm$ is the sign-extended, format-specific PC-relative offset.
- The branch condition decides whether a conditional target is selected; `jal` selects its target unconditionally.

`jalr` needs another candidate, $(R[rs1]+Imm)\mathbin{\&}\sim 1$, and a trap needs the trap-vector address. A complete processor therefore uses a next-PC multiplexer with more inputs than the two-case equation above.

The PC has two different roles in `jal`: its old value helps compute both the target $PC+Imm$ and the link value $PC+4$. Both calculations must use the same pre-edge PC. Updating the PC too early in a software-style sequence would lose that relationship; synchronous hardware instead calculates both candidates from the stable current value and commits afterward.

#### **Instruction Memory** {#instruction-memory}

Instruction memory maps an instruction address to encoded bits:

$$
Instr=IMem[PC].
$$

In the teaching datapath, this looks like a combinational read: when the PC settles, a 32-bit instruction appears after the instruction-memory delay. The instruction then fans out in parallel:

- `instr[6:0]` reaches the main decoder as the opcode;
- `instr[19:15]`, `instr[24:20]`, and `instr[11:7]` select `rs1`, `rs2`, and `rd` where those fields are meaningful;
- function fields refine the ALU or branch operation;
- scattered immediate bits reach the immediate generator.

The notation `IMem` does not require a physically separate ROM. It represents the instruction-fetch interface. A Harvard-style teaching model uses separate instruction and data interfaces so a single-cycle load can fetch an instruction and read data during the same cycle. A unified-memory implementation would need two ports, two caches, arbitration, or multiple cycles to avoid a structural conflict.

RV32I without compressed instructions has $IALIGN=32$, so legal instruction addresses are four-byte aligned. Consequently, the two least-significant PC bits are normally zero. With the compressed extension, two-byte alignment is permitted and the fetch unit must handle mixed 16-bit and 32-bit instruction boundaries.

Real instruction caches are synchronous structures with tags, hit checks, misses, permissions, and response handshakes. Treating instruction memory as a simple combinational box is therefore an abstraction chosen to expose datapath logic before introducing cache and pipeline timing.

#### **Register File** {#register-file}

The integer register file stores 32 architectural registers. A typical scalar RV32I datapath provides two read ports and one write port because instructions such as `add`, `sw`, and `beq` may need two source values but retire at most one integer destination.

The combinational reads are

$$
A=R[rs1],\qquad B=R[rs2],
$$

and the edge-triggered write is

$$
R^{+}[rd]=\begin{cases}
WD, & RegWrite=1\ \text{and}\ rd\ne 0,\\
R[rd], & \text{otherwise}.
\end{cases}
$$

- $A$ and $B$ are the two read values.
- $WD$ is the value selected by the write-back multiplexer.
- `RegWrite` is the write-enable control signal.
- The condition $rd\ne0$ preserves RISC-V's hardwired `x0=0` rule.

For `add x5, x5, x6`, the old values of `x5` and `x6` are read during the cycle; the new `x5` is captured only at the edge. This read-before-commit interpretation follows naturally from synchronous timing. Physical register arrays must still define or bypass same-address read/write behavior so the circuit realizes that architectural meaning.

The register file is not merely a storage array. Its port count strongly affects area, wiring, and delay. Adding more simultaneous instruction issue later requires more reads and writes or register banking, which is one reason wide superscalar designs have expensive register-file and bypass networks.

#### **ALU** {#alu}

The arithmetic logic unit transforms operands selected from the register file, immediate generator, or PC. It is reused conceptually for several jobs:

| Instruction purpose | ALU inputs | Operation | Result use |
|---|---|---|---|
| register arithmetic | $R[rs1]$, $R[rs2]$ | add, subtract, logic, compare | register write-back |
| immediate arithmetic | $R[rs1]$, $Imm$ | operation selected by `funct3` | register write-back |
| load/store address | $R[rs1]$, $Imm$ | addition | data-memory address |
| branch comparison | source values | subtract or dedicated comparison | taken/not-taken decision |
| PC-relative target | $PC$, $Imm$ | addition | next-PC candidate |

Some datapaths use the main ALU for branch targets and comparisons; others add dedicated hardware so operations can occur in parallel. The ISA does not choose between them.

For an $XLEN$-bit add, the datapath keeps the low $XLEN$ bits:

$$
Y=(A+B)\bmod 2^{XLEN}.
$$

Here $XLEN=32$ for RV32I. No architectural overflow flag is produced. Signed overflow and unsigned carry are interpretations that software detects when needed.

<details>
<summary>Python model: a small RV32I-style ALU</summary>

```python
MASK32 = (1 << 32) - 1


def to_signed32(value: int) -> int:
    """Interpret a 32-bit pattern as a signed two's-complement integer."""
    value &= MASK32
    return value if value < (1 << 31) else value - (1 << 32)


def alu(a: int, b: int, control: str) -> int:
    """Return the 32-bit result selected by the ALU control word."""
    a &= MASK32
    b &= MASK32

    if control == "ADD":
        result = a + b
    elif control == "SUB":
        result = a - b
    elif control == "AND":
        result = a & b
    elif control == "OR":
        result = a | b
    elif control == "SLT":
        result = int(to_signed32(a) < to_signed32(b))
    else:
        raise ValueError(f"unsupported ALU operation: {control}")

    # Physical RV32I outputs carry only the low 32 bits.
    return result & MASK32


assert alu(11, 7, "ADD") == 18
assert alu(0, 1, "SUB") == 0xFFFFFFFF
assert alu(0xFFFFFFFF, 0, "SLT") == 1  # -1 < 0 when signed
```

</details>

`ALUControl` is local control, not an ISA field. One implementation may encode ADD as `0000`, another as `101`, and both remain correct if their decoders and ALUs agree.

#### **Data Memory** {#data-memory}

Data memory is accessed only by load and store instructions in a load-store ISA. The ALU first produces an **effective address**:

$$
EA=(R[rs1]+Imm)\bmod 2^{32}.
$$

For a load, memory supplies bytes at $EA$ and the load unit extends or assembles them before write-back. For a store, `R[rs2]` supplies write data and byte-enable signals identify which byte lanes may change.

| Operation | Address input | Data direction | Architectural update |
|---|---|---|---|
| `lw rd, imm(rs1)` | ALU result | memory to processor | write a 32-bit word to `rd` |
| `lb/lbu` | ALU result | memory to processor | sign-extend or zero-extend one byte |
| `sw rs2, imm(rs1)` | ALU result | processor to memory | write four selected bytes |
| `sb` | ALU result | processor to memory | write one selected byte |

The simple chapter datapath assumes a successful aligned access finishes within the cycle. A real memory interface must also report access faults, page faults, alignment behavior, cache misses, and device side effects. Those responses matter for precise exceptions: a store must not become visible if the instruction is going to trap.

RISC-V is byte addressed. In a little-endian implementation, the least-significant byte of a word occupies the lowest address, but effective-address calculation itself is independent of endianness. Endianness controls how bytes and multi-byte values correspond, not which numeric address the ALU computes.

### **Register-Transfer-Level Descriptions** {#register-transfer-level-descriptions}

Register-transfer level, or **RTL**, describes what values combinational logic computes and which registers capture those values on a clock edge. It sits between ISA prose and individual gates.

Two notations must remain distinct:

- $Y=F(X)$ describes a combinational relationship that continuously follows its inputs after propagation delay.
- $Q^{+}\leftarrow D$ describes an enabled state update at the next edge.

For `add x5, x6, x7`, an RTL description is

$$
A=R[6],\quad B=R[7],\quad ALUResult=(A+B)\bmod 2^{32},
$$

followed at the edge by

$$
R^{+}[5]\leftarrow ALUResult,\qquad PC^{+}\leftarrow PC+4.
$$

The arrows do not imply that the register write happens before the PC update. Both updates commit from old-state inputs at the same edge. This is the hardware equivalent of building an entire next-state object before replacing the current state.

For a store, the RTL instead contains

$$
EA=R[rs1]+Imm_S,\qquad M^{+}[EA:EA+3]\leftarrow R[rs2]_{31:0},
$$

with `RegWrite=0`. For a not-taken branch, no register or memory changes and only $PC^{+}\leftarrow PC+4$ commits.

<details>
<summary>Python model: compute next state without corrupting current state</summary>

```python
from dataclasses import dataclass

MASK32 = (1 << 32) - 1


@dataclass(frozen=True)
class ArchitecturalState:
    pc: int
    registers: tuple[int, ...]


def add_transition(
    state: ArchitecturalState, rd: int, rs1: int, rs2: int
) -> ArchitecturalState:
    """Model simultaneous RTL updates for one RV32I ADD instruction."""
    # Read every source from the immutable current state.
    result = (state.registers[rs1] + state.registers[rs2]) & MASK32

    # Construct next state; do not mutate a source before all reads finish.
    next_registers = list(state.registers)
    if rd != 0:
        next_registers[rd] = result
    next_registers[0] = 0

    return ArchitecturalState(
        pc=(state.pc + 4) & MASK32,
        registers=tuple(next_registers),
    )


registers = [0] * 32
registers[5], registers[6] = 10, 8
before = ArchitecturalState(pc=0x100, registers=tuple(registers))
after = add_transition(before, rd=5, rs1=5, rs2=6)

assert before.registers[5] == 10       # current state was not overwritten
assert after.registers[5] == 18        # new state commits the result
assert after.pc == 0x104
```

</details>

This Python model does not simulate gates or time. Its value is conceptual: immutability makes simultaneous register-transfer semantics explicit and prevents accidental software-order reasoning.

### **The Single-Cycle Datapath** {#the-single-cycle-datapath}

A **single-cycle processor** completes one architectural instruction between consecutive active clock edges. Fetch, decode, register read, execute, optional data-memory access, write-back selection, and next-PC selection are logical phases inside one long combinational interval; there are no pipeline registers separating them.

The complete teaching datapath below supports the five representative paths required by the previous chapter's design recommendation: `add`, `lw`, `sw`, `beq`, and `jal`.

::: {.diagram-scroll .wide-diagram}
![A pedagogical single-cycle RV32I datapath. Black paths carry data and addresses; dashed orange paths carry control signals.](assets/single-cycle-rv32i-datapath.svg){fig-align="center" width="100%"}
:::

The datapath is assembled by sharing hardware and inserting multiplexers wherever instruction classes need different sources or destinations:

1. The PC supplies the instruction address and also feeds the `PC+4` and PC-relative target adders.
2. Instruction fields drive the register file, immediate generator, and control unit in parallel.
3. The first ALU operand normally comes from `rs1`; the second is selected from `rs2` or the immediate.
4. The ALU result can be an arithmetic answer or a data-memory address.
5. The write-back multiplexer selects an ALU result, memory data, or `PC+4` for `jal`.
6. Branch comparison and jump control choose the sequential or target next PC.

The word **single-cycle** does not mean each block takes one cycle. It means the entire register-to-register path is contained in one cycle. Combinational blocks react concurrently. For example, while the register file is being read, the immediate generator can reconstruct the offset and the control decoder can determine `ALUSrc`.

At the edge, the architectural updates can be summarized as

$$
\begin{aligned}
PC^{+} &\leftarrow NextPC,\\
R^{+}[rd] &\leftarrow Result && \text{if } RegWrite,\\
M^{+}[EA] &\leftarrow WriteData && \text{if } MemWrite.
\end{aligned}
$$

Only the enabled lines commit. A store calculates a value on the result bus but does not write a register. An `add` presents an address to data memory but does not assert memory write. Preventing unintended side effects is as important as calculating the correct result.

<details>
<summary>Python model: execute the five teaching instruction classes</summary>

```python
from dataclasses import dataclass

MASK32 = (1 << 32) - 1


@dataclass
class Machine:
    pc: int
    registers: list[int]
    words: dict[int, int]


def execute(machine: Machine, instruction: tuple) -> None:
    """Commit one abstract add/lw/sw/beq/jal instruction."""
    op, *args = instruction
    old_pc = machine.pc
    next_pc = (old_pc + 4) & MASK32

    # Calculate candidate effects first, like combinational datapath values.
    register_write: tuple[int, int] | None = None
    memory_write: tuple[int, int] | None = None

    if op == "add":
        rd, rs1, rs2 = args
        value = (machine.registers[rs1] + machine.registers[rs2]) & MASK32
        register_write = (rd, value)
    elif op == "lw":
        rd, offset, rs1 = args
        address = (machine.registers[rs1] + offset) & MASK32
        register_write = (rd, machine.words[address])
    elif op == "sw":
        rs2, offset, rs1 = args
        address = (machine.registers[rs1] + offset) & MASK32
        memory_write = (address, machine.registers[rs2])
    elif op == "beq":
        rs1, rs2, offset = args
        if machine.registers[rs1] == machine.registers[rs2]:
            next_pc = (old_pc + offset) & MASK32
    elif op == "jal":
        rd, offset = args
        register_write = (rd, (old_pc + 4) & MASK32)
        next_pc = (old_pc + offset) & MASK32
    else:
        raise ValueError(f"unsupported instruction: {op}")

    # Commit selected state changes together.
    if memory_write is not None:
        address, value = memory_write
        machine.words[address] = value & MASK32
    if register_write is not None:
        rd, value = register_write
        if rd != 0:
            machine.registers[rd] = value & MASK32
    machine.registers[0] = 0
    machine.pc = next_pc


cpu = Machine(pc=0x100, registers=[0] * 32, words={0x200C: 37})
cpu.registers[6], cpu.registers[7] = 0x2000, 7
execute(cpu, ("lw", 5, 12, 6))
assert (cpu.pc, cpu.registers[5]) == (0x104, 37)
```

</details>

The model captures architectural effects, not physical timing. Its candidate-then-commit structure mirrors the datapath discipline and will later make exception suppression easier to express.

### **Control Signal Generation** {#control-signal-generation}

Control converts encoded meaning into a **control word**: a collection of Boolean enables and multi-bit selector values applied during the current cycle. The decoder itself is combinational in a single-cycle design. It does not execute a list of commands; all output signals settle from the same instruction bits.

![Opcode decoding creates broad instruction-class signals, function fields refine the ALU operation, and branch or jump conditions select the next PC.](assets/control-decode-hierarchy.svg){fig-align="center" width="100%"}

Separating main control from ALU control reduces duplication. Loads and stores have different architectural effects but both need the ALU to add a base register and immediate. Main control can therefore request a generic ADD behavior without duplicating all function-field logic.

The names used here are pedagogical, not standardized RISC-V signal names. Another implementation may combine, split, or rename them while preserving the same architectural state transitions.

#### **Main Control** {#main-control}

The **main decoder** primarily examines the seven-bit opcode. The opcode identifies the broad class and therefore determines which immediate format, operand source, write destination, memory action, and next-PC behavior are possible.

| Instruction | Opcode | `RegWrite` | `ImmSrc` | `ALUSrc` | `MemWrite` | `ResultSrc` | `Branch` | `Jump` | `ALUOp` |
|---|---|---:|---|---|---:|---|---:|---:|---|
| R-type ALU | `0110011` | 1 | X | register | 0 | ALU | 0 | 0 | decode funct |
| `lw` | `0000011` | 1 | I | immediate | 0 | memory | 0 | 0 | ADD |
| `sw` | `0100011` | 0 | S | immediate | 1 | X | 0 | 0 | ADD |
| `beq` | `1100011` | 0 | B | register | 0 | X | 1 | 0 | compare |
| `jal` | `1101111` | 1 | J | X | 0 | PC+4 | 0 | 1 | X |

`X` means **do not care**: the output cannot affect architectural state because a later enable or multiplexer selection excludes that path. Do-not-care values give logic synthesis more freedom, but they are safe only when the gating argument is correct. For example, `ResultSrc` is irrelevant for `sw` because `RegWrite=0`.

Some diagrams include `MemRead`. A simple combinational memory can continuously produce read data, so only `MemWrite` is architecturally dangerous. A realistic cache interface needs explicit request type, size, valid, ready, fault, and byte-enable signals; its control contract is much richer.

<details>
<summary>Python model: decode RV32I opcodes into main-control signals</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class MainControl:
    reg_write: bool
    imm_src: str | None
    alu_src: str
    mem_write: bool
    result_src: str | None
    branch: bool
    jump: bool
    alu_op: str


CONTROL_BY_OPCODE = {
    0b0110011: MainControl(True,  None, "REG", False, "ALU", False, False, "FUNCT"),
    0b0000011: MainControl(True,  "I", "IMM", False, "MEM", False, False, "ADD"),
    0b0100011: MainControl(False, "S", "IMM", True,  None,  False, False, "ADD"),
    0b1100011: MainControl(False, "B", "REG", False, None,  True,  False, "COMPARE"),
    0b1101111: MainControl(True,  "J", "NA",  False, "PC4", False, True,  "NA"),
}


def decode_main(instruction: int) -> MainControl:
    opcode = instruction & 0x7F
    try:
        return CONTROL_BY_OPCODE[opcode]
    except KeyError as error:
        # Hardware would raise an illegal-instruction exception.
        raise ValueError(f"unsupported opcode 0b{opcode:07b}") from error


assert decode_main(0x007302B3).result_src == "ALU"  # add x5,x6,x7
assert decode_main(0x00C32283).result_src == "MEM"  # lw x5,12(x6)
assert decode_main(0x00732623).mem_write             # sw x7,12(x6)
```

</details>

#### **ALU Control** {#alu-control}

The main decoder knows that a load needs addition and that a branch needs comparison, but an R-type opcode alone does not distinguish `add`, `sub`, `and`, `or`, or `slt`. The **ALU decoder** combines a broad `ALUOp` with `funct3` and relevant `funct7` bits.

For the small subset:

| `ALUOp` | `funct3` | `instr[30]` | ALU action | Example |
|---|---|---:|---|---|
| ADD | X | X | ADD | `lw`, `sw` address |
| FUNCT | `000` | 0 | ADD | `add` |
| FUNCT | `000` | 1 | SUB | `sub` |
| FUNCT | `111` | X | AND | `and` |
| FUNCT | `110` | X | OR | `or` |
| FUNCT | `010` | X | signed SLT | `slt` |

`instr[30]` is enough to distinguish `add` from `sub` only after the opcode and `funct3` establish the relevant instruction family. Looking at that bit in isolation would misdecode unrelated formats.

Branch comparison deserves care. `beq` checks equality, `blt` performs a signed less-than comparison, and `bltu` performs an unsigned one. A design can reuse ALU flags or use a dedicated comparator. The latter can calculate the branch condition in parallel with the target adder, shortening the next-PC path.

<details>
<summary>Python model: hierarchical ALU decoding</summary>

```python
def decode_alu(alu_op: str, funct3: int = 0, bit30: int = 0) -> str:
    """Translate broad main control plus function bits into a local ALU action."""
    if alu_op == "ADD":
        return "ADD"
    if alu_op == "COMPARE":
        return "COMPARE"
    if alu_op != "FUNCT":
        raise ValueError(f"ALU is not used for ALUOp={alu_op}")

    if funct3 == 0b000:
        return "SUB" if bit30 else "ADD"
    if funct3 == 0b111:
        return "AND"
    if funct3 == 0b110:
        return "OR"
    if funct3 == 0b010:
        return "SLT"
    raise ValueError(f"unsupported funct3={funct3:03b}")


assert decode_alu("ADD") == "ADD"                 # lw/sw
assert decode_alu("FUNCT", 0b000, 0) == "ADD"    # add
assert decode_alu("FUNCT", 0b000, 1) == "SUB"    # sub
assert decode_alu("FUNCT", 0b111, 0) == "AND"
```

</details>

Hierarchical decoding is an implementation optimization. Synthesis may flatten the logic again, but the hierarchy gives designers a clean way to reason about instruction class first and fine operation second.

#### **Immediate Generation** {#immediate-generation}

RISC-V immediates are not always stored contiguously because register fields remain in fixed positions across formats. The immediate generator rearranges encoded bits, inserts an implicit low zero for branch and jump offsets, and sign-extends the result to $XLEN$.

![I, S, B, and J immediates use different instruction fields but share instr bit 31 as the sign bit.](assets/riscv-immediate-generation.svg){fig-align="center" width="100%"}

Using concatenation notation, the teaching subset reconstructs

$$
\begin{aligned}
Imm_I &= sext(\{instr[31:20]\}),\\
Imm_S &= sext(\{instr[31:25],instr[11:7]\}),\\
Imm_B &= sext(\{instr[31],instr[7],instr[30:25],instr[11:8],0\}),\\
Imm_J &= sext(\{instr[31],instr[19:12],instr[20],instr[30:21],0\}).
\end{aligned}
$$

- Braces mean bit concatenation from most significant to least significant.
- `sext` copies the immediate's sign bit into every higher output bit.
- The final zero in B and J immediates makes the byte offset even; it is not stored explicitly.
- `instr[31]` is the sign bit for every immediate format, allowing sign extension to begin while lower bits are routed.

For an $n$-bit encoded value $x$, sign extension to 32 bits can be expressed numerically as

$$
sext_n(x)=\begin{cases}
x, & x<2^{n-1},\\
x-2^n, & x\ge2^{n-1}.
\end{cases}
$$

The threshold $2^{n-1}$ tests the sign bit. Subtracting $2^n$ converts the unsigned bit-pattern value into its negative two's-complement interpretation.

<details>
<summary>Python model: decode I, S, B, and J immediates</summary>

```python
def bits(value: int, high: int, low: int) -> int:
    """Extract inclusive bit range [high:low]."""
    width = high - low + 1
    return (value >> low) & ((1 << width) - 1)


def sign_extend(value: int, width: int) -> int:
    sign = 1 << (width - 1)
    return (value ^ sign) - sign


def decode_immediate(instruction: int, kind: str) -> int:
    if kind == "I":
        raw, width = bits(instruction, 31, 20), 12
    elif kind == "S":
        raw = (bits(instruction, 31, 25) << 5) | bits(instruction, 11, 7)
        width = 12
    elif kind == "B":
        raw = (
            (bits(instruction, 31, 31) << 12)
            | (bits(instruction, 7, 7) << 11)
            | (bits(instruction, 30, 25) << 5)
            | (bits(instruction, 11, 8) << 1)
        )
        width = 13
    elif kind == "J":
        raw = (
            (bits(instruction, 31, 31) << 20)
            | (bits(instruction, 19, 12) << 12)
            | (bits(instruction, 20, 20) << 11)
            | (bits(instruction, 30, 21) << 1)
        )
        width = 21
    else:
        raise ValueError(f"unknown immediate kind: {kind}")
    return sign_extend(raw, width)


assert decode_immediate(0xFFF00093, "I") == -1  # addi x1,x0,-1
assert decode_immediate(0x00732623, "S") == 12  # sw x7,12(x6)
```

</details>

Immediate generation is wiring plus sign-extension logic, not arithmetic decoding software. Its delay nevertheless matters because a load or store cannot calculate an address until the immediate reaches the ALU input.

### **Tracing Instruction Classes** {#tracing-instruction-classes}

Tracing means following one instruction from its encoded fields to every value and control signal that can affect architectural state. A reliable trace answers four questions:

1. Which old-state values are read?
2. Which combinational candidates are calculated?
3. Which control signals make a path active?
4. Which state elements, if any, commit at the edge?

The following traces deliberately reuse the same datapath but activate different routes. This is the main benefit of the shared design: hardware is not duplicated for every mnemonic.

#### **R-Type Instructions** {#r-type-instructions}

Consider `add x5, x6, x7` at `PC=0x100`, with `R[6]=11` and `R[7]=7`.

![The R-type add trace reads two registers, adds them, selects the ALU result for write-back, and advances the PC.](assets/trace-r-type-add.svg){fig-align="center" width="100%"}

The opcode selects the R-type class. `funct3=000` and the relevant `funct7` bit select ADD rather than SUB. Both register outputs feed the ALU because `ALUSrc=register`. Data memory may receive the ALU result on its address wires, but `MemWrite=0`, so no memory state changes. `ResultSrc=ALU` and `RegWrite=1` route 18 to `x5`.

| Phase | Value or decision |
|---|---|
| fetch | instruction bits from `IMem[0x100]` |
| decode | `rd=5`, `rs1=6`, `rs2=7`, R-type control |
| operand read | $A=11$, $B=7$ |
| execute | $ALUResult=18$ |
| write-back | $R^{+}[5]=18$ |
| next PC | $PC^{+}=0x104$ |

The apparent sequence is a reasoning order. In hardware, fetch must precede field decoding through propagation, but register read, control decode, target calculation, and unrelated candidate paths overlap as soon as their inputs become available.

#### **Loads and Stores** {#loads-and-stores}

Loads and stores share address generation but differ in data direction and commit target. Suppose `R[6]=0x2000`, the offset is 12, and the effective address is

$$
EA=0x2000+12=0x200C.
$$

![A load reads memory and writes a register, whereas a store sends the second register value directly to memory and performs no register write-back.](assets/trace-load-store.svg){fig-align="center" width="100%"}

For `lw x5, 12(x6)`:

- `ImmSrc=I` reconstructs 12 and `ALUSrc=immediate` selects it;
- the ALU adds base and offset;
- data memory reads the word at `0x200C`;
- `ResultSrc=memory` selects the returned word;
- `RegWrite=1` commits it to `x5`.

For `sw x7, 12(x6)`:

- `ImmSrc=S` reconstructs the same numeric offset from different bit positions;
- the ALU calculates the same address;
- `R[7]` bypasses the ALU operand selection and reaches memory's write-data port;
- `MemWrite=1` commits the word;
- `RegWrite=0` prevents any register update.

<details>
<summary>Python model: little-endian word load and store paths</summary>

```python
MASK32 = (1 << 32) - 1


def store_word(memory: bytearray, address: int, value: int) -> None:
    """Model four active byte lanes for an aligned little-endian SW."""
    if address % 4 != 0:
        raise ValueError("teaching model requires word alignment")
    memory[address : address + 4] = (value & MASK32).to_bytes(4, "little")


def load_word(memory: bytearray, address: int) -> int:
    """Model the data-memory read and LW assembly path."""
    if address % 4 != 0:
        raise ValueError("teaching model requires word alignment")
    return int.from_bytes(memory[address : address + 4], "little")


registers = [0] * 32
registers[6] = 0x20
registers[7] = 0x12345678
memory = bytearray(128)

# Address ALU performs the same addition for SW and LW.
effective_address = registers[6] + 12
store_word(memory, effective_address, registers[7])
registers[5] = load_word(memory, effective_address)

assert effective_address == 0x2C
assert memory[0x2C:0x30] == bytes([0x78, 0x56, 0x34, 0x12])
assert registers[5] == 0x12345678
```

</details>

Loads usually form the single-cycle critical path because they include instruction fetch, register read, address calculation, data-memory access, and register write-back. Stores omit write-back, while ALU instructions omit data-memory access.

#### **Branches and Jumps** {#branches-and-jumps}

Control-flow instructions calculate a target and decide whether it replaces `PC+4`. They should not be reduced to 鈥渃hanging the PC鈥? `beq` needs a condition but no link value, whereas `jal` needs a link value but no condition.

![A taken BEQ selects a PC-relative target after comparison; JAL always selects its target and writes PC plus four to the link register.](assets/trace-branch-jump.svg){fig-align="center" width="100%"}

For `beq x5, x6, +16` at `PC=0x100`, the target candidate is

$$
Target=PC+Imm_B=0x100+16=0x110.
$$

The comparator calculates $BrEq=(R[5]=R[6])$. The next-PC rule is

$$
PC^{+}=\begin{cases}
Target, & Branch\land BrEq,\\
PC+4, & \text{otherwise}.
\end{cases}
$$

`Branch` says that the instruction is conditional; `BrEq` says that this particular condition is true. Keeping those meanings separate prevents an equality result from redirecting an unrelated instruction.

For `jal x1, +32`, two results are required from the old PC:

$$
R^{+}[1]=PC+4=0x104,\qquad PC^{+}=PC+Imm_J=0x120.
$$

The link lets a later `jalr x0, 0(x1)` return. `ResultSrc=PC+4` and `RegWrite=1` commit the link while `Jump=1` selects the target.

<details>
<summary>Python model: next-PC and link decisions</summary>

```python
MASK32 = (1 << 32) - 1


def beq_transition(pc: int, left: int, right: int, offset: int) -> int:
    """Select a branch target only when the equality condition holds."""
    target = (pc + offset) & MASK32
    sequential = (pc + 4) & MASK32
    return target if (left & MASK32) == (right & MASK32) else sequential


def jal_transition(pc: int, offset: int) -> tuple[int, int]:
    """Return (next_pc, link_value) calculated from the same old PC."""
    return (pc + offset) & MASK32, (pc + 4) & MASK32


assert beq_transition(0x100, 9, 9, 16) == 0x110
assert beq_transition(0x100, 9, 8, 16) == 0x104
assert jal_transition(0x100, 32) == (0x120, 0x104)
```

</details>

In the base ISA, a taken branch or jump to an illegal instruction alignment raises an exception instead of committing the normal control transfer. That requirement joins next-PC logic to the exception path discussed at the end of the chapter.

### **Critical Path and Clock-Period Limitations** {#critical-path-and-clock-period-limitations}

The clock period must be long enough for the slowest enabled register-to-register path under worst-case process, voltage, temperature, wiring, and clock conditions. A simplified constraint is

$$
T_{clk}\ge t_{clk\rightarrow q}+t_{comb,max}+t_{setup}+t_{skew}+t_{uncertainty}.
$$

- $T_{clk}$ is the period between active edges.
- $t_{clk\rightarrow q}$ is the launch register's clock-to-output delay.
- $t_{comb,max}$ is the maximum combinational path delay.
- $t_{setup}$ is the destination register's required setup time.
- $t_{skew}$ and $t_{uncertainty}$ reserve margin for clock arrival differences and variation.

For a single-cycle `lw`, the long path typically crosses the PC, instruction memory, register file, immediate/operand selection, ALU, data memory, write-back multiplexer, and destination-register setup.

![An illustrative load path totals 800 ps and forces shorter instruction paths to wait for the same clock edge.](assets/single-cycle-critical-path.svg){fig-align="center" width="100%"}

The diagram's values form a teaching estimate, not a fabrication claim. It uses

$$
T_{lw}=30+200+120+150+250+30+20=800\text{ ps}.
$$

Even if an `add` settles after 550 ps, it cannot start the next instruction early; all instructions share the 800 ps period. Therefore, for a single-cycle processor,

$$
T_{program}=N\times 1\times T_{clk},
$$

where $N$ is the number of executed instructions and the middle factor is ideal $CPI=1$. A low CPI does not guarantee a short execution time if the cycle is long.

<details>
<summary>Python model: compare instruction path delays with the global clock</summary>

```python
path_ps = {
    "add": 30 + 200 + 120 + 150 + 30 + 20,
    "lw":  30 + 200 + 120 + 150 + 250 + 30 + 20,
    "sw":  30 + 200 + 120 + 150 + 250,
    "beq": 30 + 200 + 120 + 150 + 30,
    "jal": 30 + 200 + 80 + 30,
}

clock_ps = max(path_ps.values())
unused_ps = {name: clock_ps - delay for name, delay in path_ps.items()}

assert clock_ps == 800
assert unused_ps["add"] == 250
assert unused_ps["lw"] == 0

# Every instruction still consumes exactly one 800 ps cycle.
instruction_count = 1_000
program_time_ns = instruction_count * clock_ps / 1_000
assert program_time_ns == 800
```

</details>

Duplicating adders can shorten paths by calculating `PC+4` and targets in parallel with the main ALU. Faster memory and register-file designs help too. Eventually, however, one-cycle memory and wire delay limit scaling. Multi-cycle execution shortens the clock by placing registers between operations; pipelining later overlaps those operations across instructions.

### **The Multi-Cycle Datapath** {#the-multi-cycle-datapath}

A **multi-cycle processor** divides an instruction into several clocked state transitions. Hardware can be reused because operations that occurred simultaneously in the single-cycle design now occur in different cycles. One ALU may increment the PC during fetch, calculate an effective address later, and perform arithmetic in another state.

Intermediate values would disappear when inputs change unless the design stores them. Typical internal registers include:

- `IR`: the current instruction;
- `A` and `B`: register-file read values;
- `ALUOut`: an arithmetic result, target, or effective address;
- `MDR`: data returned by memory;
- `oldPC`: the instruction address used for PC-relative calculations after the PC has advanced.

These are microarchitectural state, not additional programmer-visible registers.

![A finite-state machine sends each instruction through fetch and decode, then through instruction-specific execute, memory, and write-back states.](assets/multicycle-datapath-fsm.svg){fig-align="center" width="100%"}

An illustrative schedule is:

| Instruction class | Cycle sequence | Approximate CPI |
|---|---|---:|
| R-type / immediate ALU | IF, ID, EX-ALU, WB-ALU | 4 |
| load | IF, ID, EX-ADDR, MEM-READ, WB-LOAD | 5 |
| store | IF, ID, EX-ADDR, MEM-WRITE | 4 |
| branch | IF, ID, EX-BRANCH | 3 |
| `jal` | IF, ID, EX-JUMP | 3 |

The clock period now follows the slowest **stage**, plus internal-register overhead, rather than the whole load path. Program time remains

$$
T_{program}=N\times CPI_{avg}\times T_{clk},
$$

with

$$
CPI_{avg}=\sum_i f_i\,CPI_i.
$$

- $f_i$ is the dynamic frequency of instruction class $i$ and all frequencies sum to 1.
- $CPI_i$ is that class's number of cycles.
- A shorter $T_{clk}$ competes with a larger $CPI_{avg}$; multi-cycle is not automatically faster.

<details>
<summary>Python model: instruction mix and multi-cycle execution time</summary>

```python
cpi = {"alu": 4, "load": 5, "store": 4, "branch": 3, "jal": 3}
mix = {"alu": 0.45, "load": 0.25, "store": 0.10, "branch": 0.15, "jal": 0.05}

assert abs(sum(mix.values()) - 1.0) < 1e-12
average_cpi = sum(mix[name] * cpi[name] for name in mix)

# Illustrative comparison using the delay assumptions from this chapter.
single_cycle_ps = 800
multi_cycle_ps = 250
average_multi_instruction_ps = average_cpi * multi_cycle_ps

assert abs(average_cpi - 4.05) < 1e-12
assert average_multi_instruction_ps == 1012.5
assert average_multi_instruction_ps > single_cycle_ps
```

</details>

In this numerical example, the multi-cycle design is slower per instruction despite a much shorter clock. Its advantages are resource reuse, simpler memory timing, and a foundation for sequencing variable-latency operations. The next chapter's pipeline keeps short stages but overlaps different instructions, targeting throughput rather than merely dividing one instruction.

### **Hardwired and Microprogrammed Control** {#hardwired-and-microprogrammed-control}

A single-cycle decoder is naturally **hardwired**: Boolean logic maps instruction fields directly to control signals. A multi-cycle design also needs a state machine, but that state machine can be implemented in two broad ways.

![Hardwired logic computes a control word directly, while microprogrammed control reads a sequence of microinstructions from a control store.](assets/hardwired-vs-microprogrammed-control.svg){fig-align="center" width="100%"}

**Hardwired control** represents state transitions and outputs as gates, a programmable logic array, or synthesized RTL. It usually provides low control latency and suits regular instruction sets, but changing a complex sequence requires changing and re-verifying the logic.

**Microprogrammed control** stores microinstructions in a control memory. A microprogram counter selects one microinstruction, whose bits drive datapath enables and choose the next microaddress. One architectural instruction can expand into several internal micro-operations.

| Dimension | Hardwired control | Microprogrammed control |
|---|---|---|
| representation | Boolean logic and FSM | control store plus microsequencer |
| control latency | often lower | includes control-memory/sequencing access |
| complex sequences | can become difficult to modify | naturally expressed as microinstruction sequences |
| updates | hardware/RTL change | control-store update may be possible |
| common use | regular fast paths | complex, rare, compatibility, or corrective sequences |

The categories are not exclusive. A modern core may hardwire common instructions and invoke microcode for complex operations, assists, initialization, or post-silicon fixes. Conversely, a RISC-V implementation may be entirely hardwired, but the open ISA does not prohibit microcode.

<details>
<summary>Python model: a tiny microprogram emits control words over time</summary>

```python
MICROCODE = {
    "IF": {
        "signals": {"MemRead": 1, "IRWrite": 1, "PCWrite": 1, "ALU": "PC+4"},
        "next": "ID",
    },
    "ID": {
        "signals": {"ReadRegs": 1, "ImmGen": 1},
        "dispatch": {"lw": "EX_ADDR", "add": "EX_ALU"},
    },
    "EX_ADDR": {
        "signals": {"ALUSrc": "IMM", "ALU": "ADD", "ALUOutWrite": 1},
        "next": "MEM_READ",
    },
    "MEM_READ": {
        "signals": {"MemRead": 1, "MDRWrite": 1},
        "next": "WB_LOAD",
    },
    "WB_LOAD": {
        "signals": {"ResultSrc": "MDR", "RegWrite": 1},
        "next": "IF",
    },
}


def control_sequence(opcode_name: str) -> list[dict]:
    state = "IF"
    emitted = []
    while True:
        microinstruction = MICROCODE[state]
        emitted.append(microinstruction["signals"])
        if state == "ID":
            state = microinstruction["dispatch"][opcode_name]
        else:
            state = microinstruction["next"]
        if state == "IF":
            return emitted


lw_sequence = control_sequence("lw")
assert len(lw_sequence) == 5
assert lw_sequence[-1]["RegWrite"] == 1
```

</details>

Microcode is below the architectural instruction level. Software still observes one `lw`; the internal control sequence is an implementation detail unless timing or side channels reveal it.

### **Exceptions and Precise Architectural State** {#exceptions-and-precise-architectural-state}

An **exception** is a synchronous event caused by the current instruction, such as an illegal encoding, a misaligned target, an environment call, or a memory access fault. An **interrupt** is an asynchronous request from outside the current instruction stream, such as a timer or device event. RISC-V uses **trap** as the general transfer of control to privileged software caused by either kind of event.

A trap is **precise** when software can observe a clean instruction boundary:

1. every older instruction's architectural effects are complete;
2. the faulting instruction has made no ordinary architectural change when it must be retried or reported;
3. no younger instruction has changed architectural state;
4. trap state identifies why and where control changed.

![A precise load fault leaves older effects committed, suppresses the faulting instruction's normal write-back, records trap CSRs, and redirects the PC to the trap vector.](assets/precise-trap-state.svg){fig-align="center" width="100%"}

At machine privilege, important RISC-V trap registers include:

| CSR | Purpose during a machine-level trap |
|---|---|
| `mepc` | instruction address from which handling or resumption is defined |
| `mcause` | interrupt flag and encoded exception/interrupt cause |
| `mtval` | optional cause-specific value, often a faulting address or instruction bits |
| `mtvec` | base address and mode used to select the trap-handler entry |
| `mstatus` | records and changes privilege/interrupt-enable state |

For a synchronous fault, `mepc` normally identifies the instruction that encountered the exception. For an interrupt taken between instructions, it identifies the next not-yet-executed point from which execution can resume. Exact cause-specific rules come from the privileged architecture.

In a single-cycle core, precision is conceptually straightforward because only one instruction is active. Detect trap conditions before the edge, choose a prioritized cause, and gate ordinary side effects:

$$
RegWrite_{final}=RegWrite_{decode}\land\neg Trap,
$$

$$
MemWrite_{final}=MemWrite_{decode}\land\neg Trap,
$$

$$
PC^{+}=\begin{cases}
TrapVector, & Trap=1,\\
NormalNextPC, & Trap=0.
\end{cases}
$$

The memory system must return a fault before a load writes its destination or a store becomes visible. If multiple conditions are detected, a priority encoder must select the architecturally defined cause. Internal combinational values may be meaningless after the trap, but they are harmless because no normal write enable captures them.

<details>
<summary>Python model: suppress ordinary effects when a precise trap is selected</summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class CommitDecision:
    next_pc: int
    register_write: tuple[int, int] | None
    memory_write: tuple[int, int] | None
    mepc: int | None
    mcause: str | None
    mtval: int | None


def finalize_load(
    pc: int,
    rd: int,
    loaded_value: int,
    address: int,
    access_fault: bool,
    mtvec: int,
) -> CommitDecision:
    """Choose either the normal load commit or an atomic trap-state update."""
    if access_fault:
        return CommitDecision(
            next_pc=mtvec,
            register_write=None,       # suppress the faulting LW write-back
            memory_write=None,
            mepc=pc,
            mcause="load access fault",
            mtval=address,
        )
    return CommitDecision(
        next_pc=pc + 4,
        register_write=(rd, loaded_value),
        memory_write=None,
        mepc=None,
        mcause=None,
        mtval=None,
    )


fault = finalize_load(0x108, 5, 0, 0xDEAD0000, True, mtvec=0x80000000)
assert fault.next_pc == 0x80000000
assert fault.register_write is None
assert fault.mepc == 0x108
assert fault.mtval == 0xDEAD0000
```

</details>

Multi-cycle hardware may have changed internal registers before a fault is known, but those registers are not architectural state and can be abandoned or overwritten. Pipelining makes precision harder because several instructions coexist and younger instructions may have executed speculatively. Chapter 07 will add ordering, flushing, and commit rules that preserve the same clean boundary.

**Chapter summary.** The ISA defines architectural state transitions; the datapath supplies values and the control unit enables only the transitions permitted by the current instruction. The PC, register file, instruction interface, ALU, and data-memory interface form the core path. RTL distinguishes continuously calculated candidates from simultaneous edge-triggered updates. A single-cycle implementation shares these components through multiplexers and completes `add`, `lw`, `sw`, `beq`, or `jal` in one long cycle. Main control decodes instruction class, ALU control refines the operation, and immediate generation reconstructs scattered signed fields. Detailed traces show that R-type arithmetic writes an ALU result, loads and stores share address calculation, branches combine a target with a condition, and `jal` combines a target with a link. The load path often limits the single-cycle clock. Multi-cycle control shortens individual stages and reuses hardware at the cost of higher CPI, using either hardwired or microprogrammed sequencing. Finally, precise traps suppress ordinary writes, record cause and restart state, and redirect execution at a clean architectural boundary. Chapter 07 will overlap these same logical phases across multiple instructions and address the hazards that overlap creates.